# 🧬 NonBDNAFinder - Single-Cell Comprehensive Analysis

**Version:** 2025.1 | **Quality:** Publication-Ready | **Output:** Nature/Science Journal Standards

This notebook provides a **single-cell workflow** for complete Non-B DNA analysis:
1. Upload your FASTA sequence
2. Automatically detect 11 motif classes (24 subclasses)
3. Generate publication-quality visualizations
4. Export results in multiple formats

---

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║         NonBDNAFinder - Single-Cell Complete Analysis Pipeline                  ║
# ║              Upload → Analyze → Visualize → Export (All in One)                 ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

import warnings
warnings.filterwarnings('ignore')

# === IMPORTS ===
import os
import io
import sys
from datetime import datetime
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown, FileLink
import ipywidgets as widgets

# Ensure NonBDNAFinder modules are importable
if '.' not in sys.path:
    sys.path.insert(0, '.')

from nonbscanner import analyze_sequence
from utilities import validate_sequence

try:
    from Bio import SeqIO
    BIO_AVAILABLE = True
except ImportError:
    BIO_AVAILABLE = False
    print("⚠️ Biopython not available. Install with: pip install biopython")

# === PUBLICATION-QUALITY PLOT SETTINGS ===
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'axes.linewidth': 1.2,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': False,
})

# === NATURE-READY COLOR PALETTE (Wong 2011) ===
NATURE_COLORS = {
    'Curved_DNA': '#CC79A7',
    'G-Quadruplex': '#0072B2',
    'Z-DNA': '#882255',
    'Cruciform': '#56B4E9',
    'Triplex': '#E69F00',
    'R-Loop': '#009E73',
    'i-Motif': '#0072B2',
    'A-philic_DNA': '#CC79A7',
    'Slipped_DNA': '#E69F00',
    'Hybrid': '#888888',
    'Non-B_DNA_Clusters': '#666666'
}

# ══════════════════════════════════════════════════════════════════════════════════
#                              STEP 1: FILE UPLOAD
# ══════════════════════════════════════════════════════════════════════════════════

print("="*80)
print("🧬 NonBDNAFinder v2025.1 - Single-Cell Analysis Pipeline")
print("="*80)
print("\n📁 STEP 1: Upload your FASTA file")
print("-" * 40)

# Global storage for results
analysis_results = {'sequence': None, 'name': None, 'motifs': None, 'df': None}

# File upload widget
file_uploader = widgets.FileUpload(
    accept='.fasta,.fa,.fna,.txt',
    multiple=False,
    description='Upload FASTA:',
    button_style='primary',
    layout=widgets.Layout(width='300px')
)

# Output area
output_area = widgets.Output()

def process_and_analyze(change):
    """Process uploaded file and run complete analysis."""
    global analysis_results
    
    with output_area:
        output_area.clear_output(wait=True)
        
        if len(file_uploader.value) == 0:
            print("⏳ Waiting for file upload...")
            return
        
        # Get file content
        uploaded_file = list(file_uploader.value.values())[0]
        file_content = uploaded_file['content']
        file_name = list(file_uploader.value.keys())[0]
        
        print(f"📄 Processing: {file_name}")
        
        try:
            # Parse FASTA
            fasta_string = file_content.decode('utf-8')
            fasta_io = io.StringIO(fasta_string)
            
            if BIO_AVAILABLE:
                records = list(SeqIO.parse(fasta_io, "fasta"))
                if len(records) == 0:
                    print("❌ No valid FASTA sequences found.")
                    return
                sequence = str(records[0].seq).upper()
                seq_name = records[0].id if records[0].id else file_name
            else:
                # Simple FASTA parsing without Biopython
                lines = fasta_string.strip().split('\n')
                seq_name = lines[0].lstrip('>').split()[0] if lines[0].startswith('>') else file_name
                sequence = ''.join(l.strip() for l in lines if not l.startswith('>'))
                sequence = sequence.upper()
            
            # Validate
            is_valid, error_msg = validate_sequence(sequence)
            if not is_valid:
                print(f"❌ Invalid sequence: {error_msg}")
                return
            
            print(f"✅ Sequence loaded: {seq_name}")
            print(f"   Length: {len(sequence):,} bp")
            
            # ══════════════════════════════════════════════════════════════════
            #                    STEP 2: RUN ANALYSIS
            # ══════════════════════════════════════════════════════════════════
            print("\n" + "="*80)
            print("🔬 STEP 2: Running Non-B DNA Detection")
            print("-" * 40)
            
            start_time = datetime.now()
            motifs = analyze_sequence(sequence, seq_name)
            elapsed = (datetime.now() - start_time).total_seconds()
            
            analysis_results['sequence'] = sequence
            analysis_results['name'] = seq_name
            analysis_results['motifs'] = motifs
            
            print(f"✅ Analysis complete in {elapsed:.2f}s")
            print(f"   Speed: {len(sequence)/elapsed:,.0f} bp/s")
            print(f"   Total motifs detected: {len(motifs):,}")
            
            if not motifs:
                print("\n⚠️ No Non-B DNA motifs detected in this sequence.")
                return
            
            # Convert to DataFrame
            df = pd.DataFrame(motifs)
            analysis_results['df'] = df
            
            # ══════════════════════════════════════════════════════════════════
            #                    STEP 3: SUMMARY STATISTICS
            # ══════════════════════════════════════════════════════════════════
            print("\n" + "="*80)
            print("📊 STEP 3: Analysis Summary")
            print("-" * 40)
            
            # Class distribution
            class_counts = df['Class'].value_counts()
            print("\n📋 Motif Class Distribution:")
            for cls, count in class_counts.items():
                pct = count / len(df) * 100
                print(f"   • {cls.replace('_', ' ')}: {count:,} ({pct:.1f}%)")
            
            # Calculate coverage
            total_coverage = df['Length'].sum()
            coverage_pct = (total_coverage / len(sequence)) * 100
            density = len(motifs) / (len(sequence) / 1000)  # motifs per kb
            
            print(f"\n📈 Key Metrics:")
            print(f"   • Sequence length: {len(sequence):,} bp")
            print(f"   • Total coverage: {total_coverage:,} bp ({coverage_pct:.2f}%)")
            print(f"   • Motif density: {density:.2f} motifs/kb")
            print(f"   • Unique classes: {df['Class'].nunique()}")
            print(f"   • Unique subclasses: {df['Subclass'].nunique()}")
            
            # ══════════════════════════════════════════════════════════════════
            #                    STEP 4: VISUALIZATIONS
            # ══════════════════════════════════════════════════════════════════
            print("\n" + "="*80)
            print("📊 STEP 4: Publication-Quality Visualizations")
            print("-" * 40)
            
            # Create figure with multiple subplots
            fig = plt.figure(figsize=(14, 10))
            
            # --- Plot 1: Class Distribution Bar Chart ---
            ax1 = fig.add_subplot(2, 2, 1)
            colors = [NATURE_COLORS.get(c, '#888888') for c in class_counts.index]
            bars = ax1.barh(class_counts.index, class_counts.values, color=colors, edgecolor='white', linewidth=0.5)
            ax1.set_xlabel('Count', fontweight='bold')
            ax1.set_title('A. Motif Class Distribution', fontweight='bold', loc='left')
            ax1.invert_yaxis()
            for bar, count in zip(bars, class_counts.values):
                ax1.text(count + max(class_counts.values)*0.01, bar.get_y() + bar.get_height()/2, 
                         f'{count:,}', va='center', fontsize=8)
            
            # --- Plot 2: Genomic Track ---
            ax2 = fig.add_subplot(2, 2, 2)
            unique_classes = df['Class'].unique()
            class_to_y = {cls: i for i, cls in enumerate(unique_classes)}
            
            for _, motif in df.iterrows():
                y = class_to_y[motif['Class']]
                color = NATURE_COLORS.get(motif['Class'], '#888888')
                ax2.barh(y, motif['End'] - motif['Start'], left=motif['Start'], 
                         height=0.7, color=color, alpha=0.7, edgecolor='none')
            
            ax2.set_yticks(range(len(unique_classes)))
            ax2.set_yticklabels([c.replace('_', ' ') for c in unique_classes], fontsize=8)
            ax2.set_xlabel('Genomic Position (bp)', fontweight='bold')
            ax2.set_title('B. Genomic Motif Track', fontweight='bold', loc='left')
            ax2.set_xlim(0, len(sequence))
            
            # --- Plot 3: Length Distribution ---
            ax3 = fig.add_subplot(2, 2, 3)
            for cls in class_counts.index[:6]:  # Top 6 classes
                cls_data = df[df['Class'] == cls]['Length']
                if len(cls_data) > 5:
                    ax3.hist(cls_data, bins=20, alpha=0.6, label=cls.replace('_', ' '),
                             color=NATURE_COLORS.get(cls, '#888888'))
            ax3.set_xlabel('Motif Length (bp)', fontweight='bold')
            ax3.set_ylabel('Frequency', fontweight='bold')
            ax3.set_title('C. Length Distribution by Class', fontweight='bold', loc='left')
            ax3.legend(fontsize=7, loc='upper right', framealpha=0.9)
            
            # --- Plot 4: Score Distribution ---
            ax4 = fig.add_subplot(2, 2, 4)
            if 'Score' in df.columns:
                score_data = []
                labels = []
                for cls in class_counts.index[:6]:
                    cls_scores = df[df['Class'] == cls]['Score'].dropna()
                    if len(cls_scores) > 0:
                        score_data.append(cls_scores)
                        labels.append(cls.replace('_', ' '))
                
                if score_data:
                    bp = ax4.boxplot(score_data, labels=labels, patch_artist=True)
                    for patch, cls in zip(bp['boxes'], class_counts.index[:len(score_data)]):
                        patch.set_facecolor(NATURE_COLORS.get(cls, '#888888'))
                        patch.set_alpha(0.7)
                    ax4.set_ylabel('Score (1-3)', fontweight='bold')
                    ax4.set_title('D. Score Distribution by Class', fontweight='bold', loc='left')
                    ax4.tick_params(axis='x', rotation=45)
            
            plt.tight_layout()
            plt.savefig('NonBDNA_Analysis_Figure.png', dpi=300, bbox_inches='tight', facecolor='white')
            plt.savefig('NonBDNA_Analysis_Figure.pdf', bbox_inches='tight', facecolor='white')
            plt.show()
            
            print("\n✅ Figures saved:")
            print("   • NonBDNA_Analysis_Figure.png (300 DPI)")
            print("   • NonBDNA_Analysis_Figure.pdf (Vector)")
            
            # ══════════════════════════════════════════════════════════════════
            #                    STEP 5: DATA TABLE
            # ══════════════════════════════════════════════════════════════════
            print("\n" + "="*80)
            print("📋 STEP 5: Results Table (First 20 rows)")
            print("-" * 40)
            
            display_cols = ['Class', 'Subclass', 'Start', 'End', 'Length', 'Score', 'Strand']
            available_cols = [c for c in display_cols if c in df.columns]
            display_df = df[available_cols].head(20).copy()
            display(display_df)
            
            # ══════════════════════════════════════════════════════════════════
            #                    STEP 6: EXPORT RESULTS
            # ══════════════════════════════════════════════════════════════════
            print("\n" + "="*80)
            print("💾 STEP 6: Exporting Results")
            print("-" * 40)
            
            # Export files
            safe_name = seq_name.replace(' ', '_')[:30]
            
            # CSV
            csv_file = f'{safe_name}_NonBDNA_results.csv'
            df.to_csv(csv_file, index=False)
            print(f"   ✅ {csv_file}")
            
            # JSON
            json_file = f'{safe_name}_NonBDNA_results.json'
            df.to_json(json_file, orient='records', indent=2)
            print(f"   ✅ {json_file}")
            
            # BED
            bed_file = f'{safe_name}_NonBDNA_results.bed'
            bed_data = []
            for _, row in df.iterrows():
                bed_data.append(f"{seq_name}\t{row['Start']-1}\t{row['End']}\t{row['Class']}\t{row.get('Score', 0):.3f}\t{row.get('Strand', '+')}")
            with open(bed_file, 'w') as f:
                f.write('\n'.join(bed_data))
            print(f"   ✅ {bed_file}")
            
            print("\n" + "="*80)
            print("🎉 ANALYSIS COMPLETE!")
            print("="*80)
            print(f"\n📊 Summary: {len(motifs):,} Non-B DNA motifs detected across {df['Class'].nunique()} classes")
            print(f"📈 Genomic coverage: {coverage_pct:.2f}% | Density: {density:.2f} motifs/kb")
            print(f"\n📁 Output files generated in current directory.")
            
        except Exception as e:
            print(f"❌ Error: {str(e)}")
            import traceback
            traceback.print_exc()

# Attach event handler
file_uploader.observe(process_and_analyze, names='value')

# Display interface
display(HTML('''
<div style="background: linear-gradient(135deg, #1e40af 0%, #7c3aed 100%); 
            padding: 20px; border-radius: 12px; margin: 10px 0;">
    <h3 style="color: white; margin: 0 0 10px 0;">🧬 Upload Your FASTA File</h3>
    <p style="color: rgba(255,255,255,0.9); margin: 0; font-size: 14px;">
        Supported formats: .fasta, .fa, .fna, .txt | Max size: 200MB
    </p>
</div>
'''))
display(file_uploader)
display(output_area)

print("\n⏳ Upload a FASTA file above to begin analysis...")